<a href="https://colab.research.google.com/github/gowri246/Language-Translation-Machine/blob/main/periscope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

with open("english.txt", "r", encoding="utf-8") as f:
    english = f.readlines()

with open("malayalam.txt", "r", encoding="utf-8") as f:
    malayalam = f.readlines()

df = pd.DataFrame({
    "English": [x.strip() for x in english],
    "Malayalam": [x.strip() for x in malayalam]
}).head(8000) # Limit to first 8000 sentences

df.to_csv("translation_dataset.csv", index=False, encoding="utf-8")

print("Dataset created successfully!")

Dataset created successfully!


In [ ]:
df.head()

,English,Malayalam
0,A large freight train sits in a train station.,ഒരു വലിയ ചരക്ക് ട്രെയിൻ ഒരു ട്രെയിൻ സ്റ്റേഷനിൽ...
1,People boat down a river with flags strung acr...,ആളുകൾ ഒരു നദിക്കരയിൽ പതാകകൾ പതിച്ചിട്ടുണ്ട്.
2,A set of railway tracks as seen from a train.,ട്രെയിനിൽ നിന്ന് കാണുന്നതുപോലെ ഒരു കൂട്ടം റെയി...
3,A canoe with a motor speeding down a river.,ഒരു നദിയിലൂടെ മോട്ടോർ വേഗതയുള്ള ഒരു പീരങ്കി.
4,A yellow van with people trying to display som...,ശൂന്യമായ സ്ക്വയറിൽ എന്തെങ്കിലും പ്രദർശിപ്പിക്ക...


## 1. Install and Import Libraries

First, we need to install the `transformers` library from Hugging Face, which provides pre-trained models and tools for NLP tasks. We'll also install `sentencepiece` for tokenization, `accelerate` for optimized training, and `datasets` for efficient data handling.

In [ ]:
pip install transformers sentencepiece accelerate datasets -q

## 2. Load and Prepare the Dataset

Now, we'll load the `translation_dataset.csv` into a Hugging Face `Dataset` object and split it into training and testing sets. This makes it easier to work with the data for model training.

In [ ]:
from datasets import load_dataset, DatasetDict

# Load the dataset from the CSV file
dataset = load_dataset('csv', data_files='translation_dataset.csv')

# Split the dataset into training and testing sets
train_test_split = dataset['train'].train_test_split(test_size=0.1)

datasets = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test'],
})

print(f"Training set size: {len(datasets['train'])}")
print(f"Test set size: {len(datasets['test'])}")
display(datasets['train'][0])

Generating train split: 0 examples [00:00, ? examples/s]

Training set size: 7200
Test set size: 800


{'English': 'a cat is sitting on top of the fridge',
 'Malayalam': 'ഫ്രിഡ്ജിന് മുകളിൽ ഒരു പൂച്ച ഇരിക്കുന്നു'}

## 3. Load Tokenizer and Model

We will use a pre-trained MarianMT model (`Helsinki-NLP/opus-mt-en-ml`) for English to Malayalam translation. This model is a good starting point for machine translation tasks. We load both its tokenizer and the model itself.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

# Define the model checkpoint
model_checkpoint = "Helsinki-NLP/opus-mt-en-ml"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Load the model with pre-trained weights (instead of from scratch)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded (with pre-trained weights): {model.__class__.__name__}")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Tokenizer loaded: MarianTokenizer
Model loaded (with pre-trained weights): MarianMTModel


## 4. Preprocess the Data

Before we can train the model, we need to tokenize the text data. This involves converting the sentences into numerical representations that the model can understand. We'll define a tokenization function that takes a batch of examples, applies the tokenizer to both the English and Malayalam texts, and returns the tokenized inputs.

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    # Tokenize English input
    model_inputs = tokenizer(examples["English"], max_length=max_input_length, truncation=True)

    # Tokenize targets (Malayalam) directly for MarianMT models
    labels = tokenizer(examples["Malayalam"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing function to the entire dataset
tokenized_datasets = datasets.map(preprocess_function, batched=True)

print("Tokenized datasets created successfully!")
display(tokenized_datasets['train'][0])

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenized datasets created successfully!


{'English': 'a cat is sitting on top of the fridge',
 'Malayalam': 'ഫ്രിഡ്ജിന് മുകളിൽ ഒരു പൂച്ച ഇരിക്കുന്നു',
 'input_ids': [13, 7939, 17, 7729, 50, 4731, 5, 4, 1636, 4315, 1587, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [9, 1, 9, 1, 9, 3867, 9, 1, 9, 1, 0]}

## 5. Define Data Collator

To efficiently handle varying sequence lengths, we use a `DataCollatorForSeq2Seq`. This collator will pad the input and target sequences to the maximum length within each batch, ensuring consistent input sizes for the model during training.

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Create a data collator for sequence-to-sequence tasks
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

print("Data collator initialized successfully!")

Data collator initialized successfully!


## 6. Configure Training Arguments and Initialize Trainer

We define `TrainingArguments` to specify training parameters like output directory, learning rate, and batch size. Then, we initialize the `Seq2SeqTrainer` with the model, training arguments, tokenizer, data collator, and datasets. Combining these steps ensures all necessary objects are defined in sequence.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch", # Removed due to TypeError in current environment
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=25, # Increased number of epochs for better training
    predict_with_generate=True,
    fp16=True, # Enable mixed precision training
    push_to_hub=False, # Set to True if you want to push your model to Hugging Face Hub
)

print("Training arguments configured successfully!")

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

print("Trainer initialized successfully!")

Training arguments configured successfully!
Trainer initialized successfully!


## 6. Configure Training Arguments

We need to define `TrainingArguments` to specify various training parameters such as the output directory, learning rate, batch size, number of epochs, and evaluation strategy. These arguments guide the training process.

## 7. Initialize Trainer

The `Seq2SeqTrainer` simplifies the training and evaluation loop for sequence-to-sequence models. We initialize it with the model, training arguments, tokenizer, data collator, and the tokenized datasets.

## 8. Train the Model

Now, we can start the training process by calling the `train()` method on the `trainer` object. This will fine-tune the MarianMT model on your English-Malayalam dataset.

In [ ]:
# Start training
trainer.train()

print("Training complete!")

Step,Training Loss
500,1.346999
1000,0.499058
1500,0.446278
2000,0.414836
2500,0.387196
3000,0.369419
3500,0.356040
4000,0.341996
4500,0.328824
5000,0.319866


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


## 9. Evaluate the Model

After training, it's important to evaluate the model's performance on the test set. We'll use the `evaluate()` method to get metrics on how well the model translates unseen examples. This evaluation might not include BLEU score calculation by default but provides loss metrics.

In [ ]:
# Evaluate the model
eval_results = trainer.evaluate()

print(f"Evaluation results: {eval_results}")

Training Loss,Validation Loss,Step
0.248417,0.404292,11250


Evaluation results: {'eval_loss': 0.4042924642562866}


### 10. Sample Translation

Let's test our trained model with a few English sentences and see how it translates them into Malayalam.

In [ ]:
import torch

# Determine the device to use (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the selected device
model.to(device)

def translate_sentence(sentence):
    # Tokenize the input English sentence and move to the device
    inputs = tokenizer(sentence, return_tensors="pt", max_length=max_input_length, truncation=True).to(device)

    # Generate translation using the model
    translated_tokens = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_target_length
    )

    # Decode the generated tokens back to Malayalam text
    translated_sentence = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_sentence

# Interactive usage:
while True:
    english_input = input("Enter an English sentence (or 'q' to quit): ")
    if english_input.lower() == 'q':
        break
    malayalam_translation = translate_sentence(english_input)
    print(f"English: {english_input}")
    print(f"Malayalam: {malayalam_translation}\n")

Enter an English sentence (or 'q' to quit): how are you
English: how are you
Malayalam: സുഖമല്ലേ?

Enter an English sentence (or 'q' to quit): what are you doing
English: what are you doing
Malayalam: നീയെന്താ ചെയ്യുന്നത്?

Enter an English sentence (or 'q' to quit): when are you going
English: when are you going
Malayalam: നീ എപ്പോഴാ പോകുന്നത്?

Enter an English sentence (or 'q' to quit): messi is the best player in the world
English: messi is the best player in the world
Malayalam: ലോകത്തിലെ ഏറ്റവും മികച്ച കളിക്കാരനാണ് ടെറി.

Enter an English sentence (or 'q' to quit): do you eat breakfast today
English: do you eat breakfast today
Malayalam: നീ ഇന്ന് പ്രാതൽ കഴിക്കുന്നുണ്ടോ?

Enter an English sentence (or 'q' to quit): kasi is the most beautiful student in the college
English: kasi is the most beautiful student in the college
Malayalam: കോളേജിലെ ഏറ്റവും സുന്ദരിയായ വിദ്യാർത്ഥിയാണ് കാസി.

Enter an English sentence (or 'q' to quit): all the girls in  the TIST college wants arun
English: 